# MoneyPrinterV2 - Cloud Edition

Run MoneyPrinterV2 on Google Colab's free infrastructure.

**Features:**
- 🚀 **Free GPU/CPU**: Faster video rendering than local CPU.
- 🌐 **Public API**: Expose the REST API via Ngrok.
- 🤖 **Interactive CLI**: Run the standard terminal interface.

In [ ]:
#@title 1. Initialize Workspace
#@markdown Clone the repository and enter the directory.
import os

if not os.path.exists("MoneyPrinterV2"):
  !git clone https://github.com/Ogak-AI/MoneyPrinterV2.git
  %cd MoneyPrinterV2
else:
  %cd MoneyPrinterV2
  !git pull

print("Current Directory:", os.getcwd())

In [ ]:
#@title 2. Install System Dependencies
#@markdown Installs FFmpeg, ImageMagick, Firefox, and Xvfb (for headless browser).

print("Installing system dependencies... (This may take 2-3 minutes)")
!apt-get update -qq
!apt-get install -y firefox-esr ffmpeg imagemagick xvfb libpango-1.0-0 libharfbuzz0b libpangoft2-1.0-0 > /dev/null

# Configure ImageMagick policy to allow text/image operations
!sed -i 's/domain="path" rights="none" pattern="@\*"/domain="path" rights="read|write" pattern="@\*"/g' /etc/ImageMagick-6/policy.xml

print("System dependencies installed.")

In [ ]:
#@title 3. Install Python Dependencies
#@markdown Installs python libraries and `pyngrok` for tunneling.

!pip install -r requirements.txt
!pip install pyngrok
print("Python dependencies installed.")

In [ ]:
#@title 4. Install GeckoDriver
#@markdown Downloads the correct GeckoDriver for the installed Firefox version.

import requests
import tarfile
import subprocess

print("Installing GeckoDriver...")
# Fetch latest release tag
response = requests.get("https://api.github.com/repos/mozilla/geckodriver/releases/latest")
tag_name = response.json()["tag_name"]
download_url = f"https://github.com/mozilla/geckodriver/releases/download/{tag_name}/geckodriver-{tag_name}-linux64.tar.gz"

!wget -q {download_url} -O geckodriver.tar.gz
!tar -xzf geckodriver.tar.gz
!mv geckodriver /usr/local/bin/
!rm geckodriver.tar.gz
!chmod +x /usr/local/bin/geckodriver

print("GeckoDriver installed.")

In [ ]:
#@title 5. Configuration
#@markdown Set your API keys here. Leave empty to use defaults/env vars.

import os
import json

config_content = {
  "verbose": True,
  "firefox_profile": "",
  "headless": True,
  "ollama_base_url": "http://localhost:11434",
  "ollama_model": "",
  "twitter_language": "English",
  "nanobanana2_api_base_url": "https://generativelanguage.googleapis.com/v1beta",
  "nanobanana2_api_key": "",  # ENTER YOUR GEMINI API KEY HERE
  "nanobanana2_model": "gemini-3.1-flash-image-preview",
  "nanobanana2_aspect_ratio": "9:16",
  "threads": 2,
  "zip_url": "",
  "is_for_kids": False,
  "google_maps_scraper": "https://github.com/gosom/google-maps-scraper/archive/refs/tags/v0.9.7.zip",
  "email": {
    "smtp_server": "smtp.gmail.com",
    "smtp_port": 587,
    "username": "",
    "password": ""
  },
  "google_maps_scraper_niche": "",
  "scraper_timeout": 300,
  "outreach_message_subject": "I have a question...",
  "outreach_message_body_file": "outreach_message.html",
  "stt_provider": "local_whisper",
  "whisper_model": "base",
  "whisper_device": "cuda",  # Use Colab GPU
  "whisper_compute_type": "int8",
  "assembly_ai_api_key": "",
  "tts_voice": "Jasper",
  "font": "bold_font.ttf",
  "imagemagick_path": "/usr/bin/convert",
  "script_sentence_length": 4
}

# Write config.json
with open("config.json", "w") as f:
    json.dump(config_content, f, indent=4)

print("Config saved to config.json")

In [ ]:
#@title 6. Run API with Ngrok Tunnel
#@markdown Starts the API server and exposes it via a public URL.
#@markdown You need a free Ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTH_TOKEN = "" #@param {type:"string"}

import os
from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    # Kill existing tunnels
    ngrok.kill()
    # Open tunnel
    public_url = ngrok.connect(8000).public_url
    print(f"🚀 Public API URL: {public_url}")
    print(f"   Docs available at: {public_url}/docs")

# Start Xvfb (Virtual Display) and API
print("Starting API...")
!sh -c "Xvfb :99 -screen 0 1280x1024x24 & export DISPLAY=:99 && python src/api.py"

In [ ]:
#@title Alternative: Run Interactive CLI
#@markdown Run this cell if you prefer the interactive terminal menu instead of the API.

!sh -c "Xvfb :99 -screen 0 1280x1024x24 & export DISPLAY=:99 && python src/main.py"